In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

# Combined main-text figure: two original simulations run independently.
# Models: stable diffusive KS + minimal CMIPS; stable autocrine KS + autocrine CMIPS.
# Original equations, parameters, initial conditions, clipping/rescaling, dt, grid and seed are preserved.
# Panel (a): Pearson correlation for both simulations, t = 60--120.
# Panel (b): t = 120 maps; rows: phi_1 and phi_1 - <phi_1>; columns: Autocrine and Minimal.

def laplacian(field, spacing):
    return (np.roll(field, -1, axis=0) + np.roll(field, 1, axis=0) + np.roll(field, -1, axis=1) + np.roll(field, 1, axis=1) - 4.0 * field) / spacing**2

def gradient_x(field, spacing):
    return (np.roll(field, -1, axis=0) - np.roll(field, 1, axis=0)) / (2.0 * spacing)

def gradient_y(field, spacing):
    return (np.roll(field, -1, axis=1) - np.roll(field, 1, axis=1)) / (2.0 * spacing)

def compute_mu(phi, phi_m_value, pe_r):
    phi_safe = np.clip(phi, 1e-6, phi_m_value - 1e-6)
    term1 = np.log(phi_safe)
    term2 = 1.0 - 2.0 * phi_safe - 0.3 * phi_safe**2
    term3 = -(4.0 / np.pi) * pe_r * phi_m_value * (np.log(1.0 - phi_safe / phi_m_value) - phi_safe / phi_m_value - phi_safe)
    return term1 + term2 + term3

def compute_pearson(phi1, phi2):
    x = phi1.ravel()
    y = phi2.ravel()
    sigma1 = np.std(x)
    sigma2 = np.std(y)
    if sigma1 < 1e-12 or sigma2 < 1e-12:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

# 1. Stable diffusive KS + minimal-signalling CMIPS

def run_minimal_simulation():
    Nx, Ny = 100, 100
    dx = 1.0
    dt = 0.001
    T = 120000
    plot_interval = 10000
    np.random.seed(1)
    phi_m = 0.9
    phi0_total = 0.8
    phi10 = phi0_total / 2
    phi20 = phi0_total / 2
    D_phi1 = 2
    chi1 = 1
    M2 = 2
    chi2 = 1
    Pe_R = 0.001
    kappa = 5.0
    D_c = 1.0
    a2 = 0.1
    k = 0.1
    phi1 = phi10 + 0.02 * (np.random.rand(Nx, Ny) - 0.5)
    phi2 = phi20 + 0.02 * (np.random.rand(Nx, Ny) - 0.5)
    phi1 = np.clip(phi1, 0.0, None)
    phi2 = np.clip(phi2, 0.0, None)
    c0 = a2 * phi20 / k
    c = c0 + 0.02 * (np.random.rand(Nx, Ny) - 0.5)
    c = np.clip(c, 0.0, None)
    steps = [0]
    pearson_history = [compute_pearson(phi1, phi2)]
    for t in range(1, T + 1):
        phi_total = phi1 + phi2
        phi_total_safe = np.clip(phi_total, 1e-6, phi_m - 1e-6)
        grad_c_x = gradient_x(c, dx)
        grad_c_y = gradient_y(c, dx)
        grad_phi1_x = gradient_x(phi1, dx)
        grad_phi1_y = gradient_y(phi1, dx)
        flux1_x = D_phi1 * grad_phi1_x - chi1 * phi1 * grad_c_x
        flux1_y = D_phi1 * grad_phi1_y - chi1 * phi1 * grad_c_y
        mu = compute_mu(phi_total_safe, phi_m, Pe_R)
        mu_eff = mu - kappa * laplacian(phi_total_safe, dx)
        grad_mu_x = gradient_x(mu_eff, dx)
        grad_mu_y = gradient_y(mu_eff, dx)
        flux2_x = M2 * phi2 * grad_mu_x - chi2 * phi2 * grad_c_x
        flux2_y = M2 * phi2 * grad_mu_y - chi2 * phi2 * grad_c_y
        div_flux1 = gradient_x(flux1_x, dx) + gradient_y(flux1_y, dx)
        div_flux2 = gradient_x(flux2_x, dx) + gradient_y(flux2_y, dx)
        phi1_new = phi1 + dt * div_flux1
        phi2_new = phi2 + dt * div_flux2
        lap_c = laplacian(c, dx)
        production = a2 * phi2
        degradation = k * c
        c_new = c + dt * (D_c * lap_c + production - degradation)
        if (not np.all(np.isfinite(phi1_new)) or not np.all(np.isfinite(phi2_new)) or not np.all(np.isfinite(c_new))):
            raise FloatingPointError(f"Non-finite value detected in minimal-signalling simulation at step {t}")

        # IMPORTANT: preserve original minimal-signalling evolution exactly: no clipping and no density rescaling
        phi1 = phi1_new
        phi2 = phi2_new
        c = c_new
        if t % plot_interval == 0:
            steps.append(t)
            pearson_history.append(compute_pearson(phi1, phi2))
    steps = np.asarray(steps)
    times = steps * dt
    return {"times": times, "pearson": np.asarray(pearson_history), "phi1": phi1.copy(), "phi2": phi2.copy(),}

# 2. Stable autocrine KS + autocrine CMIPS

def run_autocrine_simulation():
    Nx, Ny = 100, 100
    dx = 1.0
    dt = 0.001
    T = 120000
    plot_interval = 10000
    np.random.seed(1)
    phi_m = 0.9
    phi0_total = 0.8
    phi10 = phi0_total / 2
    phi20 = phi0_total / 2
    D_phi1 = 2.0
    M2 = 2.0
    Pe_R = 0.001
    kappa = 5.0
    chi1 = 1.0
    chi2 = 1.0
    D_c = 1.0
    a1 = 0.1
    a2 = 0.1
    k1 = 0.1
    k2 = 0.1
    phi1 = phi10 + 0.02 * (np.random.rand(Nx, Ny) - 0.5)
    phi2 = phi20 + 0.02 * (np.random.rand(Nx, Ny) - 0.5)
    phi1 = np.clip(phi1, 0.0, None)
    phi2 = np.clip(phi2, 0.0, None)
    uptake0 = k1 * phi10 + k2 * phi20
    if uptake0 <= 0:
        raise ValueError("The homogeneous uptake rate must be positive.")
    c0 = (a1 * phi10 + a2 * phi20) / uptake0

    # IMPORTANT: preserve original autocrine initial condition exactly
    c = 0.02 * (np.random.rand(Nx, Ny) - 0.5)
    c = np.clip(c, 0.0, None)
    steps = [0]
    pearson_history = [compute_pearson(phi1, phi2)]
    for t in range(1, T + 1):
        phi_total = phi1 + phi2
        phi_total_clipped = np.clip(phi_total, 1e-6, phi_m - 1e-6)
        grad_c_x = gradient_x(c, dx)
        grad_c_y = gradient_y(c, dx)
        grad_phi1_x = gradient_x(phi1, dx)
        grad_phi1_y = gradient_y(phi1, dx)
        flux1_x = D_phi1 * grad_phi1_x - chi1 * phi1 * grad_c_x
        flux1_y = D_phi1 * grad_phi1_y - chi1 * phi1 * grad_c_y
        mu = compute_mu(phi_total_clipped, phi_m, Pe_R)
        mu_eff = mu - kappa * laplacian(phi_total_clipped, dx)
        grad_mu_x = gradient_x(mu_eff, dx)
        grad_mu_y = gradient_y(mu_eff, dx)
        flux2_x = M2 * phi2 * grad_mu_x - chi2 * phi2 * grad_c_x
        flux2_y = M2 * phi2 * grad_mu_y - chi2 * phi2 * grad_c_y
        div_flux1 = gradient_x(flux1_x, dx) + gradient_y(flux1_y, dx)
        div_flux2 = gradient_x(flux2_x, dx) + gradient_y(flux2_y, dx)
        phi1_new = phi1 + dt * div_flux1
        phi2_new = phi2 + dt * div_flux2
        lap_c = laplacian(c, dx)
        production = a1 * phi1 + a2 * phi2
        uptake = (k1 * phi1 + k2 * phi2) * c
        c_new = c + dt * (D_c * lap_c + production - uptake)

        # IMPORTANT: preserve original clipping
        phi1 = np.clip(phi1_new, 0.0, None)
        phi2 = np.clip(phi2_new, 0.0, None)
        c = np.clip(c_new, 0.0, None)

        # IMPORTANT: preserve original total-density rescaling
        phi_total = phi1 + phi2
        exceed = phi_total > phi_m
        if np.any(exceed):
            scale = phi_m / (phi_total[exceed] + 1e-12)
            phi1[exceed] *= scale
            phi2[exceed] *= scale
        if t % plot_interval == 0:
            steps.append(t)
            pearson_history.append(compute_pearson(phi1, phi2))
    steps = np.asarray(steps)
    times = steps * dt
    return {"times": times, "pearson": np.asarray(pearson_history), "phi1": phi1.copy(), "phi2": phi2.copy(),}

# Run both simulations
print("Running minimal-signalling simulation...")
minimal = run_minimal_simulation()
print("Running autocrine simulation...")
autocrine = run_autocrine_simulation()
print("Both simulations complete.")

# Final-state fluctuation maps
minimal_delta_phi1 = minimal["phi1"] - np.mean(minimal["phi1"])
autocrine_delta_phi1 = autocrine["phi1"] - np.mean(autocrine["phi1"])
delta_abs_max = max(np.max(np.abs(minimal_delta_phi1)), np.max(np.abs(autocrine_delta_phi1)))

# Main-text combined figure
fig = plt.figure(figsize=(8.8, 9.0))
outer = GridSpec(nrows=2, ncols=1, figure=fig, height_ratios=[1.0, 1.9], left=0.12, right=0.90, bottom=0.07, top=0.97, hspace=0.25)

# Panel a: Pearson correlation
ax_a = fig.add_subplot(outer[0])
minimal_mask = (minimal["times"] >= 60) & (minimal["times"] <= 120)
autocrine_mask = (autocrine["times"] >= 60) & (autocrine["times"] <= 120)
ax_a.plot(minimal["times"][minimal_mask], minimal["pearson"][minimal_mask], marker="o", linewidth=2, markersize=5, label="Minimal signalling")
ax_a.plot(autocrine["times"][autocrine_mask], autocrine["pearson"][autocrine_mask], marker="s", linewidth=2, markersize=5, label="Autocrine signalling")
ax_a.axhline(0, linestyle=":", linewidth=1)
ax_a.set_xlim(60, 120)
ax_a.set_ylim(-1.05, 1.05)
ax_a.set_xticks([60, 70, 80, 90, 100, 110, 120])
ax_a.set_xlabel("Time")
ax_a.set_ylabel(r"Pearson correlation, $r_{12}$")
ax_a.legend(frameon=False, loc="best")
ax_a.text(-0.10, 1.03, "a", transform=ax_a.transAxes, fontsize=16, fontweight="bold")

# Panel b: t = 120 maps
lower = GridSpecFromSubplotSpec(nrows=2, ncols=3, subplot_spec=outer[1], width_ratios=[1.0, 1.0, 0.055], hspace=0.08, wspace=0.10)
ax_auto_phi1 = fig.add_subplot(lower[0, 0])
ax_min_phi1 = fig.add_subplot(lower[0, 1])
ax_auto_delta = fig.add_subplot(lower[1, 0])
ax_min_delta = fig.add_subplot(lower[1, 1])
cax_phi1 = fig.add_subplot(lower[0, 2])
cax_delta = fig.add_subplot(lower[1, 2])

# Same phi1 display scale as the original scripts
phi1_vmin = 0.0
phi1_vmax = 0.9 / 2.0
im_auto_phi1 = ax_auto_phi1.imshow(autocrine["phi1"], origin="lower", cmap="Reds", vmin=phi1_vmin, vmax=phi1_vmax, interpolation="nearest", aspect="equal")
im_min_phi1 = ax_min_phi1.imshow(minimal["phi1"], origin="lower", cmap="Reds", vmin=phi1_vmin, vmax=phi1_vmax, interpolation="nearest", aspect="equal")
im_auto_delta = ax_auto_delta.imshow(autocrine_delta_phi1, origin="lower", cmap="coolwarm", vmin=-delta_abs_max, vmax=delta_abs_max, interpolation="nearest", aspect="equal")
im_min_delta = ax_min_delta.imshow(minimal_delta_phi1, origin="lower", cmap="coolwarm", vmin=-delta_abs_max, vmax=delta_abs_max, interpolation="nearest", aspect="equal")
ax_auto_phi1.set_title("Autocrine signalling", pad=8)
ax_min_phi1.set_title("Minimal signalling", pad=8)

for ax in [ax_auto_phi1, ax_min_phi1, ax_auto_delta, ax_min_delta]:
    ax.set_xticks([])
    ax.set_yticks([])
    
ax_auto_phi1.set_ylabel(r"$\phi_1$", rotation=0, labelpad=26, va="center", fontsize=13)
ax_auto_delta.set_ylabel(r"$\phi_1-\langle\phi_1\rangle$", rotation=0, labelpad=52, va="center", fontsize=13)
ax_auto_phi1.text(-0.28, 1.08, "b", transform=ax_auto_phi1.transAxes, fontsize=16, fontweight="bold")
cbar_phi1 = fig.colorbar(im_min_phi1, cax=cax_phi1)
cbar_phi1.set_label(r"$\phi_1$")
cbar_delta = fig.colorbar(im_min_delta, cax=cax_delta)
cbar_delta.set_label(r"$\phi_1-\langle\phi_1\rangle$")
fig.text(0.505, 0.535, r"$t=120$", ha="center", va="bottom")

plt.show()
